# FinReq AI — Fine-Tuning Qwen2.5 on Financial Requirement Engineering (QLoRA + Unsloth)

This notebook fine-tunes **Qwen2.5-7B-Instruct** (or 3B fallback) using Unsloth on Google Colab (T4 / A100 GPU).
It exports a GGUF Q4_K_M model ready for local inference with Ollama.

### 1. Mount Google Drive (for Checkpoint & Model Persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/finreq/outputs'
DATA_DIR = '/content/drive/MyDrive/finreq/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Outputs and Data directories initialized at: {OUTPUT_DIR}')

### 2. Install Unsloth and Dependencies

In [ ]:
%%capture
!pip install --no-deps unsloth
!pip install "torch" "transformers" "datasets" "accelerate" "bitsandbytes" "trl" "peft" "xformers"


### 3. Load Base Model in 4-bit Precision

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096
DTYPE = None  # None for auto-detection
LOAD_IN_4BIT = True

# Primary: Qwen2.5-7B-Instruct (Fallback if OOM: unsloth/Qwen2.5-3B-Instruct-bnb-4bit)
MODEL_NAME = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print('Base model loaded successfully.')

### 4. Configure LoRA Adapters (Targeting All Projection Layers)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('LoRA adapter configured.')

### 5. Load and Format Dataset (Train & Validation)

In [ ]:
from datasets import load_dataset

train_path = os.path.join(DATA_DIR, 'train.jsonl')
val_path = os.path.join(DATA_DIR, 'val.jsonl')

if not os.path.exists(train_path):
    train_path = 'train.jsonl'
    val_path = 'val.jsonl'

dataset = load_dataset('json', data_files={'train': train_path, 'val': val_path})

def formatting_prompts_func(examples):
    convos = examples['messages']
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {'text': texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f'Loaded {len(dataset["train"])} training examples and {len(dataset["val"])} validation examples.')

### 6. Train with SFTTrainer (Loss on Assistant Tokens Only)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.03,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=100,
        optim='paged_adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=OUTPUT_DIR,
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<|im_start|>user\n',
    response_part='<|im_start|>assistant\n',
)

trainer_stats = trainer.train()
print('Training complete!')

### 7. Save LoRA Adapter & Export to GGUF (Q4_K_M)

In [ ]:
# 1. Save 16-bit LoRA adapter
adapter_path = os.path.join(OUTPUT_DIR, 'lora_model')
model.save_pretrained_merged(adapter_path, tokenizer, save_method='lora')
print(f'LoRA adapter saved to {adapter_path}')

# 2. Export to GGUF Q4_K_M for Ollama CPU inference
model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method='q4_k_m')
print(f'GGUF export finished in: {OUTPUT_DIR}')